# convert

Turns Hive someone else wrote, one query or a batch of scripts, into your own code: each table the scripts write becomes a Saved table in your Dev Database, with its Table folder, and what only you can decide of it, a Decision, answered in the decisions cell. Run each step's cell in turn, or, in their place, the last cell, project.run_workflow(...), which runs every step left in turn, up to the first only you can do, and says where it stopped. First fill in the names cell below and save the notebook: each call reads it, and so do project.check(), which names the next step, and project.do_next(), the cell before the last, which runs only the steps that change files, up to the first it can't.

In [ ]:
import pathlib
import sys
sys.path.insert(0, str(pathlib.Path.cwd().parents[2]))  # the Project: three up

from sqlglot_composer import admin, hive_settings, project, run_execute, run_query, show_hive
from warehouse import query, execute

In [ ]:
NAME = 'team_fails'  # the batch's name, plain lowercase, such as "weekly" (or, for the last cell alone, a list of batches converted already)
FILES = ["weekly_fails.hql"]  # the scripts' files, from this folder, such as ["daily.hql"], or [] to read HIVE
HIVE = None  # the Hive as text, such as """SELECT ...""", or None to read FILES
DATABASE = 'default'  # the database of a table the Hive names without one
RENAME = {"mart.team_fails": "team_fails"}  # a name of your own for a table's copy in dev, by the table's full name, such as {"mart.team_fails": "team_fails"} for mart_dev.team_fails: left out, a copy is <table>_<its database>, such as mart_dev.team_fails_mart
STAGING = {}  # each staging table convert names, kept or folded, such as {"ops_reports.failed": "keep"}: left out, a temporary or DROPped one is folded, any other kept

In [ ]:
DECISIONS = {"mart_dev.team_fails": {"location": "/data/team/mart_dev.db/team_fails"}}  # your answer to each open Decision, by the table's name as
# project.check() writes it, such as
# {"mart_dev.weekly_ops": {"location": "/data/team/mart_dev.db/weekly",
#                          "key": ["job_id"], "types": {"fails": "bigint"}}}
# "key": None says no columns name one row. project.check() names each one open;
# a Key proposed from a query is made, but an answer here changes it

## 1. Convert the scripts: each table they write a Saved table in dev

It is a call to the warehouse, yours to run.

In [ ]:
project.convert(NAME, HIVE, files=FILES, query=query, database=DATABASE,
                rename=RENAME, staging=STAGING)

## 2. Answer each table's open Decisions in the decisions cell

It is a decision, yours to make.

## 3. Write each answer into its table's Table reference

It changes only files, so project.do_next() runs it for you.

In [ ]:
project.write_decisions(DECISIONS)

## 4. Create each table in dev

It is a call to the warehouse, yours to run.

In [ ]:
for table in project.batch_tables(NAME):
    run_execute(project.load(table).create(), execute)

## 5. Seed each table that reads itself, once, from its original

It is a call to the warehouse, yours to run.

In [ ]:
run_execute(hive_settings("dynamic_partitions"), execute)
for table in project.batch_tables(NAME, seeded=True):
    run_execute(project.load(table).seed(), execute)

## 6. Write each table in dev

It is a call to the warehouse, yours to run.

In [ ]:
project.run_tables(tables=project.batch_tables(NAME), execute=execute)

## 7. Compare each table with its original

It is a call to the warehouse, yours to run.

In [ ]:
project.compare_batch(NAME, query=query)

In [ ]:
project.do_next()

In [ ]:
# Every step left, in place of their cells, up to the first only you can do.
# Add dry_run=True to see the Hive it would send, sending nothing through your
# execute: help(project.run_workflow) says the steps it still runs:
project.run_workflow("convert", query=query, execute=execute)